In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.scale import FuncScale

from pollen_worker.pollen_utils import get_clients_population_dict

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "bold"

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha

In [ ]:
datasets_list = ["openimage", "google_speech", "shakespeare", "reddit"]

In [ ]:
map_datasets = {
    "openimage": "Open Image",
    "google_speech": "Google Speech",
    "shakespeare": "Shakespeare",
    "reddit": "Reddit",
}

In [ ]:
cid_samples_dicts = {}
for dataset in datasets_list:
    print(dataset)
    cid_samples_dicts[dataset] = get_clients_population_dict(
        name=dataset, batch_size=1, seed=1337
    )

In [ ]:
dfs = []
fig, ax = plt.subplots(figsize=(7, 5))
for dataset in datasets_list:
    print(dataset)
    data = random.sample(
        cid_samples_dicts[dataset].items(),
        k=min(10000, len(cid_samples_dicts[dataset])),
    )  # Subsampling Reddit
    # data = cid_samples_dicts[dataset].items() # All of Reddit
    df = pd.DataFrame(data, columns=["client_id", "samples"])
    df["dataset"] = map_datasets[dataset]
    print(df.samples.describe())
    print(df.samples.skew())
    print(f"Max number of samples: {np.max(df.samples)}")
    df.samples = df.samples.apply(lambda x: x / np.max(df.samples))
    print(df.samples.describe())
    line = df.samples.plot.kde(
        ax=ax, ind=1_000, label="", linewidth=2.0, bw_method="silverman"
    )
    ax.hist(
        df.samples,
        bins=50,
        density=True,
        label=f"{map_datasets[dataset]}",
        color=line.get_lines()[-1].get_color(),
        alpha=0.4,
    )
    dfs.append(df)
df = pd.concat(dfs)
y_label = ax.set_ylabel("Density of number of clients")
y_label.set_weight("bold")
x_label = ax.set_xlabel("Number of samples (normalized)")
x_label.set_weight("bold")
ax.legend().set_title("")
ax.set_xlim(0.0029, 1.0)
x_axis = ax.get_xaxis()
ax.set_xscale(FuncScale(x_axis, functions=(lambda x: np.sqrt(x + 1e-16), lambda x: x)))
ax.grid()
plt.savefig("datasets_kde_client_distr.pdf", format="pdf", dpi=800, bbox_inches="tight")
plt.show()